# Evaluating translations with BLEU - Review

<div style="border:1px solid #ccc; border-radius:6px; padding:12px;">

<br>
<b>About</b><br><br>

This notebook is derived from the following notebook, with modifications and extensions: https://github.com/AI-Engineering-bootcamp/ai-eng-nbs-public/blob/master/bleu-evaluation-202504.ipynb
</div>

## Initial Setup

To install the dependencies, uncomment and run the lines below:



In [1]:
# !pip install -q transformers==4.46.0 sympy==1.13.3 deep-translator evaluate==0.4.3

<br>

## Intro

In this notebook, we will use the BLEU metric to compare the quality of two different approaches for performing translations.

I will translate a few lines from the beginning of this chapter from English to Spanish. My translations will be taken as the reference translations. In other words, they will be used as the basis upon which the quality of the automatic translations will be determined.


<br>

**TASK: Perform your own translations to your mother tongue so you can evaluate the quality of the results** 

<br>

In [2]:
#Sentences to Translate.
sentences = [
    "In the previous chapters, you've mainly seen how to work with OpenAI models, and you've had a very practical introduction to Hugging Face's open-source models, the use of embeddings, vector databases, and agents.",
    "These have been very practical chapters in which I've tried to gradually introduce concepts that have allowed you, or at least I hope so, to scale up your knowledge and start creating projects using the current technology stack of large language models."
    ]

In [3]:
#Spanish Translation References.
reference_translations = [
    ["En los capítulos anteriores has visto mayoritariamente como trabajar con los modelos de OpenAI, y has tenido una introducción muy práctica a los modelos Open Source de Hugging Face, al uso de embeddings, las bases de datos vectoriales, los agentes."],
    ["Han sido capítulos muy prácticos en los que he intentado ir introduciendo conceptos que te han permitido, o eso espero, ir escalando en tus conocimientos y empezar a crear proyectos usando el stack tecnológico actual de los grandes modelos de lenguaje."]
    ]

We will perform the first translation using the NLLB model, a small model specialized in performing translations, which we will retrieve from Hugging Face.

Below, we load the tokenizer and model for **NLLB-200-distilled-600M** (`facebook/nllb-200-distilled-600M`) from Hugging Face.

NLLB ("No Language Left Behind") is a family of translation models released by Meta AI, trained to translate directly between 200 languages without going through English as an intermediate step. The `distilled-600M` version we use here is a smaller, distilled variant (600 million parameters) of the original larger model, making it fast enough to run locally while still producing good quality translations.

Once loaded, we build a `translation` pipeline from the model and tokenizer, specifying the source language (`eng_Latn`, English) and target language (`spa_Latn`, Spanish). We then use that pipeline to translate our sentences into Spanish.

In [4]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline

model_id = "facebook/nllb-200-distilled-600M"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSeq2SeqLM.from_pretrained(model_id)

/Users/luis/Desktop/ironhack_june26/1_ai_eng_lectures/.venv/langchain-v0.2.x/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


When creating the pipeline, we pass the source language and the target language of the translation to it.

In [5]:
translator = pipeline('translation', model=model, tokenizer=tokenizer,
                        src_lang="eng_Latn", tgt_lang="spa_Latn")

Hardware accelerator e.g. GPU is available in the environment, but no `device` argument is passed to the `Pipeline` object. Model will be on CPU.


In [6]:
import os

os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'

translations_nllb = [] # we'll store here the translations

for text in sentences:
  translation = ""
  translation = translator(text)
  print ("\n\nText to translate: " + text)
  print ("Translation: " + translation[0]['translation_text'])

  #Add the summary to summaries list
  translations_nllb += translation[0].values()



Text to translate: In the previous chapters, you've mainly seen how to work with OpenAI models, and you've had a very practical introduction to Hugging Face's open-source models, the use of embeddings, vector databases, and agents.
Translation: En los capítulos anteriores, han visto principalmente cómo trabajar con modelos OpenAI, y han tenido una introducción muy práctica a los modelos de código abierto de Hugging Face, el uso de embebidos, bases de datos vectoriales y agentes.


Text to translate: These have been very practical chapters in which I've tried to gradually introduce concepts that have allowed you, or at least I hope so, to scale up your knowledge and start creating projects using the current technology stack of large language models.
Translation: Estos han sido capítulos muy prácticos en los que he intentado introducir gradualmente conceptos que han permitido, o al menos espero que lo hagan, ampliar sus conocimientos y comenzar a crear proyectos utilizando la tecnologí

<br>

Now we have the translations stored in the list 'translations_nllb'.

In [7]:
translations_nllb

['En los capítulos anteriores, han visto principalmente cómo trabajar con modelos OpenAI, y han tenido una introducción muy práctica a los modelos de código abierto de Hugging Face, el uso de embebidos, bases de datos vectoriales y agentes.',
 'Estos han sido capítulos muy prácticos en los que he intentado introducir gradualmente conceptos que han permitido, o al menos espero que lo hagan, ampliar sus conocimientos y comenzar a crear proyectos utilizando la tecnología actual de los modelos de lenguaje grande.']

<br>

## Create Translations with Google Traslator.

As a second source for translations, we will use the Google Translator API.

In [8]:
from deep_translator import GoogleTranslator

In [9]:
translator_google = GoogleTranslator(source="en", target="es")

In [10]:
translations_google = []

for text in sentences:
  print ("\n\nText to translate: " + text)
  translation = ""
  translation = translator_google.translate(text)

  #Add the summary to summaries list
  translations_google.append(translation)
  print ("Translation: " + translation)



Text to translate: In the previous chapters, you've mainly seen how to work with OpenAI models, and you've had a very practical introduction to Hugging Face's open-source models, the use of embeddings, vector databases, and agents.
Translation: En los capítulos anteriores, vio principalmente cómo trabajar con modelos OpenAI y tuvo una introducción muy práctica a los modelos de código abierto de Hugging Face, el uso de incrustaciones, bases de datos vectoriales y agentes.


Text to translate: These have been very practical chapters in which I've tried to gradually introduce concepts that have allowed you, or at least I hope so, to scale up your knowledge and start creating projects using the current technology stack of large language models.
Translation: Estos han sido capítulos muy prácticos en los que he intentado introducir gradualmente conceptos que te han permitido, o al menos eso espero, ampliar tus conocimientos y empezar a crear proyectos utilizando la tecnología actual de gra

<br>

In this list, we have the translations created by Google.

In [11]:
translations_google

['En los capítulos anteriores, vio principalmente cómo trabajar con modelos OpenAI y tuvo una introducción muy práctica a los modelos de código abierto de Hugging Face, el uso de incrustaciones, bases de datos vectoriales y agentes.',
 'Estos han sido capítulos muy prácticos en los que he intentado introducir gradualmente conceptos que te han permitido, o al menos eso espero, ampliar tus conocimientos y empezar a crear proyectos utilizando la tecnología actual de grandes modelos de lenguaje.']

<br>

## Evaluate translations with BLEU

We will use the BLEU implementation from the Evaluate library by Hugging Face.

In [12]:
import evaluate
bleu = evaluate.load('bleu')

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [13]:
results_nllb = bleu.compute(predictions=translations_nllb, references=reference_translations)


To obtain the metrics, we pass the translated text and the reference text to the BLEU function.

Note that the translated text is a list of translations:
["Translation1", "Translation2"]

Whereas the reference texts are a list of lists of text. This allows for providing multiple references per translation:

[["reference1 Translation1", "reference2 Translation1"],
["reference2 Translation2", "reference2 Translation2"]]


In [14]:
results_google = bleu.compute(predictions=translations_google, references=reference_translations)

In [15]:
print(results_nllb)

{'bleu': 0.3686324165619373, 'precisions': [0.7159090909090909, 0.47674418604651164, 0.30952380952380953, 0.18292682926829268], 'brevity_penalty': 0.988700685876667, 'length_ratio': 0.9887640449438202, 'translation_length': 88, 'reference_length': 89}


In [16]:
print(results_google)

{'bleu': 0.44975901966417653, 'precisions': [0.7710843373493976, 0.5679012345679012, 0.4177215189873418, 0.2987012987012987], 'brevity_penalty': 0.9302618655343314, 'length_ratio': 0.9325842696629213, 'translation_length': 83, 'reference_length': 89}


### How to interpret the BLEU score

BLEU returns a `bleu` value between 0 and 1 (sometimes reported as 0-100), measuring how closely the predicted translation's word/n-gram sequences match the reference translation. A score of **1.0 (or 100)** would mean a perfect, word-for-word match with the reference; **0.0** means no overlap at all.

In practice, for a single-reference translation, absolute scores tend to look low even for decent translations, because BLEU is strict about matching exact wording and phrase order. As a rough guide:

- **~0.40-0.60**: Considered a high-quality translation, close in wording to the reference. Understandable, mostly correct, but small differences in word choice or phrasing are common.
- **~0.20-0.40**: A reasonable, understandable translation, but with noticeable differences in wording/style from the reference.
- **Below ~0.20**: Significant differences from the reference; the translation may still be understandable, but it diverges a lot in word choice, phrasing, or structure.

So a BLEU score like **0.52** doesn't mean "52% correct" in a strict sense — it means the translation shares a good amount of matching words/phrases with the reference, but isn't an exact match. There can be many valid ways to translate the same sentence, and BLEU only rewards similarity to *this specific* reference, so a lower score doesn't necessarily mean a bad translation.

This is exactly why comparing BLEU scores alongside your own reading of the translations (as we do in the Conclusions below) is important: BLEU gives you a quick, automatic signal, but human judgment is still needed to properly assess translation quality.

## Conclusions

On top of the BLEU scores comparing nllb and googletrans, examine the output sentences visually. Does your personal opinion match the BLEU score? Which model do you think did better? 

This is why evaluating the quality of an LLM output is very difficult, and a subjective, personal inspection is always required.